# Do crack classifiers survive a change of surface? — Kaggle runner

Wraps `bash scripts/run_all.sh <data-root>` with preflight checks. It does **not** change the method or any file in `src/`.

**Before running (Session options panel, right side):**
1. **Accelerator → GPU** (T4 x2 or P100). Needs a phone-verified Kaggle account.
2. **Internet → On.** Needed once per session: torchvision downloads the ImageNet weights for ResNet50 / EfficientNet-B0 from `download.pytorch.org`. (Also requires phone verification.)
3. **Add Input → SDNET2018** (any Kaggle copy of the dataset; it must contain the folders `D/CD D/UD W/CW W/UW P/CP P/UP`). Check its license and cite it (see README).
4. **Add the code**: upload `crack-domain-shift.tar.gz` (or the unpacked folder) as a private Kaggle *Dataset* and add it as an input here. Alternatively set `REPO_URL` below.

For the long run, use **Save Version → Save & Run All (Commit)** so it keeps running if your browser disconnects. Outputs in `/kaggle/working` appear on the version's Output tab.

In [ ]:
import os, sys, shutil, subprocess, tarfile, urllib.request
from pathlib import Path

# ---------------- settings you may need to edit ----------------
DATA_ROOT = None    # e.g. "/kaggle/input/sdnet2018" ; None = auto-detect (folder that directly contains D/, W/, P/)
REPO_URL = None     # e.g. "https://github.com/<you>/crack-domain-shift" ; None = use the uploaded code dataset
SEED = 42           # run_all.sh takes the seed as its 2nd argument
REQUIRE_GPU = True
REQUIRE_INTERNET = True
# ----------------------------------------------------------------
WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")

## 1. Preflight: GPU and Internet

In [ ]:
import torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__)
gpu = torch.cuda.is_available()
print("CUDA available:", gpu, "|", torch.cuda.get_device_name(0) if gpu else "no GPU")
if REQUIRE_GPU and not gpu:
    raise RuntimeError("No GPU. Session options -> Accelerator -> GPU, then restart the session.")

try:
    req = urllib.request.Request("https://download.pytorch.org/models/resnet50-11ad3fa6.pth", method="HEAD")
    urllib.request.urlopen(req, timeout=15)
    net = True
except Exception as e:
    net = False
    print("Cannot reach download.pytorch.org:", type(e).__name__, e)
print("Internet for pretrained weights:", net)
if REQUIRE_INTERNET and not net:
    raise RuntimeError("Internet is off. Session options -> Internet -> On (phone verification required), "
                       "then restart. Without it the ImageNet weights cannot be downloaded.")

## 2. Find the SDNET2018 data root

In [ ]:
def find_data_root():
    hits = []
    for dp, dn, _ in os.walk(INPUT):
        if Path(dp).name in {"CD", "UD", "CW", "UW", "CP", "UP"}:
            dn[:] = []                      # do not descend into the image folders
            continue
        if Path(dp).name == "D" and (Path(dp) / "CD").is_dir() and (Path(dp) / "UD").is_dir():
            hits.append(Path(dp).parent)
    return hits

if DATA_ROOT is None:
    hits = find_data_root()
    if not hits:
        print("Top of /kaggle/input:", [p.name for p in INPUT.iterdir()])
        raise RuntimeError("No D/CD + D/UD folders found. Is SDNET2018 added as an input? Set DATA_ROOT by hand.")
    if len(hits) > 1:
        print("Several candidates, using the first:", hits)
    DATA_ROOT = str(hits[0])
print("DATA_ROOT =", DATA_ROOT)

# count images per class folder; every folder must be non-empty
EXT = {".jpg", ".jpeg", ".png"}
counts = {}
for s, (u, c) in {"D": ("UD", "CD"), "W": ("UW", "CW"), "P": ("UP", "CP")}.items():
    for d in (u, c):
        p = Path(DATA_ROOT) / s / d
        counts[f"{s}/{d}"] = sum(1 for f in os.scandir(p) if Path(f.name).suffix.lower() in EXT) if p.is_dir() else None
print(counts)
assert all(counts.values()), f"missing or empty class folder(s): {counts}"

## 3. Get the code into /kaggle/working and run the unit tests

In [ ]:
REPO = WORK / "crack-domain-shift"
if not (REPO / "src" / "train.py").exists():
    if REPO_URL:
        subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
    else:
        folders = [p.parent.parent for p in INPUT.rglob("train.py") if p.parent.name == "src"]
        tars = list(INPUT.rglob("crack-domain-shift*.tar.gz"))
        if folders:
            shutil.copytree(folders[0], REPO)
        elif tars:
            with tarfile.open(tars[0]) as t:     # archive contains a top-level crack-domain-shift/ folder
                try:
                    t.extractall(WORK, filter="data")
                except TypeError:            # older Python builds without the 'filter' argument
                    t.extractall(WORK)
        else:
            raise RuntimeError("Code not found. Add the code dataset as an input, or set REPO_URL.")
assert (REPO / "scripts" / "run_all.sh").exists(), "unexpected archive layout"
os.chdir(REPO)
print("cwd =", Path.cwd())

In [ ]:
!python -m pytest -q 2>&1 | tail -5

## 4. Look at the real split before training (read-only)

How many **distinct photographs** end up in each split matters for how much to trust the confidence intervals later:
the bootstrap resamples *images*, but images from one photograph are strongly correlated.

In [ ]:
import pandas as pd
sys.path.insert(0, str(Path.cwd()))
from src.data import build_manifest, check_no_leakage, describe
df = build_manifest(DATA_ROOT, seed=SEED)
check_no_leakage(df)
pd.set_option("display.width", 200)
print(describe(df).to_string())
print("\ndistinct photograph groups per surface / split:")
print(df.groupby(["surface", "split"])["group"].nunique().unstack()[["train", "val", "test"]])

## 5. Quick real-data smoke test (about a minute; writes to a throw-away folder)

In [ ]:
!python -m src.train --data-root "{DATA_ROOT}" --model resnet50 --source D --seed {SEED} \
    --epochs 1 --max-per-class 64 --out /kaggle/working/_smoke/results --ckpt-dir /kaggle/working/_smoke/ckpt 2>&1 | tail -8
shutil.rmtree("/kaggle/working/_smoke", ignore_errors=True)

## 6. The full run

Exactly `bash scripts/run_all.sh <data-root> <seed>`: 2 models x 3 source surfaces, then evaluation and occlusion maps.
Output is also saved to `run_all.log`. A crash in any run stops the whole script; if that happens, fix the cause and re-run
(finished runs are retrained, there is no resume).

In [ ]:
!PYTHONUNBUFFERED=1 bash scripts/run_all.sh "{DATA_ROOT}" {SEED} 2>&1 | tee run_all.log

## 7. Check that everything was produced

In [ ]:
res = Path("results")
preds = sorted((res / "preds").glob("*.csv"))
print(len(preds), "prediction files (expected 6 for 2 models x 3 sources):")
for p in preds: print("  ", p.name)
for f in ["summary.csv", "gap.csv", "mcnemar.csv", "heatmap_resnet50.png", "heatmap_efficientnet_b0.png"]:
    print(("ok      " if (res / f).exists() else "MISSING ") + f)
print("occlusion:", [p.name for p in (res / "occlusion").glob("*.png")])
assert len(preds) == 6 and (res / "summary.csv").exists(), "run did not finish; read run_all.log"

## 8. Digest of the numbers

`scripts/results_digest.py` only reformats `summary.csv`, `gap.csv`, `mcnemar.csv` and `history/*.json`. It adds plain-language flags
(for example "CI includes 0") and computes nothing new. Read `docs/READING_RESULTS.md` before drawing conclusions.

In [ ]:
!python scripts/results_digest.py --results results

## 9. Figures

In [ ]:
from IPython.display import Image, display
for p in sorted(Path("results").glob("heatmap_*.png")) + sorted(Path("results/occlusion").glob("*.png")):
    print(p); display(Image(str(p)))

## 10. Bundle for download

In [ ]:
shutil.make_archive("/kaggle/working/results_bundle", "zip", ".", "results")
shutil.copy("run_all.log", "/kaggle/working/run_all.log") if Path("run_all.log").exists() else None
print("wrote /kaggle/working/results_bundle.zip (checkpoints/ stays in the repo folder; ~hundreds of MB, download only if needed)")